# Developer
This final notebook is designed to give you some impression how executorlib works internally and how to debug executorlib.

## Data Transfer
Transferring a large amount of data between two processes requires additional resources so it is helpful to measure the data transferred between the frontend and backend process. This is achieved by setting the `log_obj_size` parameter to `True`:

In [1]:
from executorlib import SingleNodeExecutor

with SingleNodeExecutor(log_obj_size=True) as exe:
    future = exe.submit(sum, [1, 1])
    print(future.result())

Send dictionary of size: 101
Received dictionary of size: 59
Send dictionary of size: 69
Received dictionary of size: 58


2


## Write Log Files
Libraries like executorlib are commonly used to sample a large parameter space, in this case it is possible that out of a large number of parameters one combination throws an error. This error can be logged in a file which also contains the function and input parameters using the `"error_log_file"` parameter in the `resource_dict`. This allows to change the log file on a per-function bases.

In [2]:
from executorlib import SingleNodeExecutor

In [3]:
def my_funct(i, j): 
    if i == 2 and j == 2:
        raise ValueError()
    else: 
        return i * j + i + j

A try and except statement is added to prevent the jupyter notebook from crashing:

In [4]:
with SingleNodeExecutor(resource_dict={"error_log_file": "error.out"}) as exe:
    future_lst = []
    for i in range(4):
        for j in range(4):
            future_lst.append(exe.submit(my_funct, i=i, j=j))
    try:
        print([f.result() for f in future_lst])
    except ValueError:
        pass

The content of the log file is a basic text file, so it can be read with any kind of file utility. The important part is that the log file contains not only the error message but in addition also the function name and the input parameters in the case kwargs: `{'i': 2, 'j': 2}` which helps for future debugging of the sampling function.

In [5]:
with open("error.out") as f:
    content = f.readlines()

In [6]:
content

['function: <function my_funct at 0x7fd2361cf740>\n',
 'args: ()\n',
 "kwargs: {'i': 2, 'j': 2}\n",
 'Traceback (most recent call last):\n',
 '  File "/srv/conda/envs/notebook/lib/python3.13/site-packages/executorlib/backend/interactive_serial.py", line 56, in main\n',
 '    output = call_funct(input_dict=input_dict, funct=None, memory=memory)\n',
 '  File "/srv/conda/envs/notebook/lib/python3.13/site-packages/executorlib/standalone/interactive/backend.py", line 33, in call_funct\n',
 '    return funct(input_dict["fn"], *input_dict["args"], **input_dict["kwargs"])\n',
 '  File "/srv/conda/envs/notebook/lib/python3.13/site-packages/executorlib/standalone/interactive/backend.py", line 22, in funct\n',
 '    return args[0].__call__(*args[1:], **kwargs)\n',
 '           ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^\n',
 '  File "/tmp/ipykernel_2748/3167739528.py", line 3, in my_funct\n',
 'ValueError\n',
 'function: <function my_funct at 0x7fdbccaff740>\n',
 'args: ()\n',
 "kwargs: {'i': 2, 'j': 2

## TestClusterExecutor
While the `SingleNodeExecutor` internally behaves very similar to the `FluxJobExecutor` and `SlurmJobExecutor` the `FluxClusterExecutor` and `SlurmClusterExecutor` behave very different as they use the file system to exchange information rather than socket-based communication. This can lead to complications when it comes to debugging. To address this challenge executorlib provides the `TestClusterExecutor` which can be executed on a local workstation just like the `SingleNodeExecutor` but in the background it uses the same file based communication like the `SlurmClusterExecutor` and the `FluxClusterExecutor`:

In [7]:
from executorlib.api import TestClusterExecutor

with TestClusterExecutor(cache_directory="test") as exe:
    future = exe.submit(sum, [1, 1])
    print(future.result())

2


## External Executables
On extension beyond the submission of Python functions is the communication with an external executable. This could be any kind of program written in any programming language which does not provide Python bindings so it cannot be represented in Python functions. 

### Subprocess
If the external executable is called only once, then the call to the external executable can be represented in a Python function with the [subprocess](https://docs.python.org/3/library/subprocess.html) module of the Python standard library. In the example below the shell command `echo test` is submitted to the `execute_shell_command()` function, which itself is submitted to the `Executor` class.

In [8]:
from executorlib import SingleNodeExecutor

In [9]:
def execute_shell_command(
    command: list, universal_newlines: bool = True, shell: bool = False
):
    import subprocess

    return subprocess.check_output(
        command, universal_newlines=universal_newlines, shell=shell
    )

In [10]:
with SingleNodeExecutor() as exe:
    future = exe.submit(
        execute_shell_command,
        ["echo", "test"],
        universal_newlines=True,
        shell=False,
    )
    print(future.result())

test



### Interactive
The more complex case is the interaction with an external executable during the run time of the executable. This can be implemented with executorlib using the block allocation `block_allocation=True` feature. The external executable is started as part of the initialization function `init_function` and then the indivdual functions submitted to the `Executor` class interact with the process which is connected to the external executable. 

Starting with the definition of the executable, in this example it is a simple script which just increases a counter. The script is written in the file `count.py` so it behaves like an external executable, which could also use any other progamming language. 

In [11]:
count_script = """\
def count(iterations):
    for i in range(int(iterations)):
        print(i)
    print("done")


if __name__ == "__main__":
    while True:
        user_input = input()
        if "shutdown" in user_input:
            break
        else:
            count(iterations=int(user_input))
"""

with open("count.py", "w") as f:
    f.writelines(count_script)

The connection to the external executable is established in the initialization function `init_function` of the `Executor` class. By using the [subprocess](https://docs.python.org/3/library/subprocess.html) module from the standard library two process pipes are created to communicate with the external executable. One process pipe is connected to the standard input `stdin` and the other is connected to the standard output `stdout`. 

In [12]:
def init_process():
    import subprocess

    return {
        "process": subprocess.Popen(
            ["python", "count.py"],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            universal_newlines=True,
            shell=False,
        )
    }

The interaction function handles the data conversion from the Python datatypes to the strings which can be communicated to the external executable. It is important to always add a new line `\n` to each command send via the standard input `stdin` to the external executable and afterwards flush the pipe by calling `flush()` on the standard input pipe `stdin`.  

In [13]:
def interact(shell_input, process, lines_to_read=None, stop_read_pattern=None):
    process.stdin.write(shell_input)
    process.stdin.flush()
    lines_count = 0
    output = ""
    while True:
        output_current = process.stdout.readline()
        output += output_current
        lines_count += 1
        if stop_read_pattern is not None and stop_read_pattern in output_current:
            break
        elif lines_to_read is not None and lines_to_read == lines_count:
            break
    return output

Finally, to close the process after the external executable is no longer required it is recommended to define a shutdown function, which communicates to the external executable that it should shutdown. In the case of the `count.py` script defined above this is achieved by sending the keyword `shutdown`. 

In [14]:
def shutdown(process):
    process.stdin.write("shutdown\n")
    process.stdin.flush()

With these utility functions is to possible to communicate with any kind of external executable. Still for the specific implementation of the external executable it might be necessary to adjust the corresponding Python functions. Therefore this functionality is currently limited to developers and not considered a general feature of executorlib. 

In [15]:
with SingleNodeExecutor(
    max_workers=1,
    init_function=init_process,
    block_allocation=True,
) as exe:
    future = exe.submit(
        interact, shell_input="4\n", lines_to_read=5, stop_read_pattern=None
    )
    print(future.result())
    future_shutdown = exe.submit(shutdown)
    print(future_shutdown.result())

0
1
2
3
done

None
